In [1]:
!git clone https://github.com/expaetra/CM3070_final_project.git
%cd CM3070_final_project

fatal: destination path 'CM3070_final_project' already exists and is not an empty directory.
/content/CM3070_final_project


In [2]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from urllib.parse import quote

The dataset is extended by collecting up to 500 papers per category for each year from 2021 to 2025. In this iteration, titles are also collected to examine whether they provide additional signal for classification.

### Scraper

In [3]:
taxonomy = {
    'cs.LG': {'field': 'Machine Learning', 'discipline': 'Artificial Intelligence'},
    'cs.CV': {'field': 'Computer Vision', 'discipline': 'Computer Imaging and Vision'},
    'cs.AI': {'field': 'General Artificial Intelligence', 'discipline': 'Artificial Intelligence'},
    'cs.CL': {'field': 'Natural Language Processing', 'discipline': 'Artificial Intelligence'},
    'cs.IT': {'field': 'Information Theory', 'discipline': 'Communication'},
    'cs.RO': {'field': 'General Robotics', 'discipline': 'Robotics'},
    'cs.CR': {'field': 'Cryptography', 'discipline': 'Cybersecurity'},
    'cs.HC': {'field': 'General Human-Computer Interaction', 'discipline': 'Human-Computer Interaction'},
    'cs.DS': {'field': 'Data Structures and Algorithms', 'discipline': 'Theory of Computation'},
    'cs.DC': {'field': 'Distributed Computer Systems', 'discipline': 'Distributed Systems'},
    'cs.CY': {'field': 'Computers and Society', 'discipline': 'Social Computing'},
    'cs.NI': {'field': 'General Computer Networks', 'discipline': 'Computer Networks'},
    'cs.SE': {'field': 'General Software Engineering', 'discipline': 'Software Engineering'},
    'cs.IR': {'field': 'General Information Retrieval', 'discipline': 'Information Retrieval'},
    'cs.SI': {'field': 'Social Networks', 'discipline': 'World Wide Web'},
    'cs.SD': {'field': 'Sound and Signal Processing', 'discipline': 'Multimedia'},
    'cs.LO': {'field': 'Formal Logic', 'discipline': 'Artificial Intelligence'},
    'cs.NE': {'field': 'Neural Networks', 'discipline': 'Artificial Intelligence'},
    'cs.DM': {'field': 'Discrete Mathematics', 'discipline': 'Theory of Computation'},
    'cs.GT': {'field': 'Game Theory', 'discipline': 'Artificial Intelligence'},
    'cs.CC': {'field': 'Computational Complexity', 'discipline': 'Theory of Computation'},
    'cs.MA': {'field': 'Multiagent Systems', 'discipline': 'Artificial Intelligence'},
    'cs.DB': {'field': 'Database Systems', 'discipline': 'Database Systems'},
    'cs.CE': {'field': 'Computational Engineering', 'discipline': 'Computational Science'},
    'cs.PL': {'field': 'Programming Languages', 'discipline': 'Computer Programming'},
    'cs.MM': {'field': 'General Multimedia', 'discipline': 'Multimedia'},
    'cs.CG': {'field': 'Computational Geometry', 'discipline': 'Computer Graphics'},
    'cs.AR': {'field': 'Hardware Architecture', 'discipline': 'Computer Hardware'},
    'cs.ET': {'field': 'Emerging Technologies', 'discipline': 'Emerging Technologies'},
    'cs.DL': {'field': 'Digital Libraries', 'discipline': 'World Wide Web'},
    'cs.FL': {'field': 'Formal Languages and Automata Theory', 'discipline': 'Theoretical Computer Science'},
    'cs.PF': {'field': 'Performance', 'discipline': 'Computer Systems'},
}

In [4]:
#  years included in the scraping process

years = [2021, 2022, 2023, 2024, 2025]
years

[2021, 2022, 2023, 2024, 2025]

In [5]:
# Parameters
MAX_RESULTS_PER_YEAR = 500
WAIT_TIME = 6
BATCH_SIZE = 50
MAX_RETRIES = 3

# Scraper for title + abstract by year
def scrape_arxiv(category, year, max_results=MAX_RESULTS_PER_YEAR, wait=WAIT_TIME):
    records = []
    start = 0
    ns = {'atom': 'http://www.w3.org/2005/Atom'}

    start_date = f"{year}01010000"
    end_date = f"{year}12312359"

    while len(records) < max_results:
        current_batch_size = min(BATCH_SIZE, max_results - len(records))

        params = {
            "search_query": f"cat:{category} AND submittedDate:[{start_date} TO {end_date}]",
            "start": start,
            "max_results": current_batch_size,
            "sortBy": "submittedDate",
            "sortOrder": "descending"
        }

        success = False

        for attempt in range(MAX_RETRIES):
            try:
                time.sleep(wait)

                response = requests.get(
                    "https://export.arxiv.org/api/query",
                    params=params,
                    timeout = 60
                )
                response.raise_for_status()
                if not response.content.strip():
                    raise ValueError("Empty response")

                root = ET.fromstring(response.content)
                success = True
                break

            except requests.HTTPError as e:
                status = e.response.status_code

                # retry if common errors occur
                if status == 429:
                    print(f"429 for {category}, {year} at start={start},attempt {attempt+1}/{MAX_RETRIES}")
                    print("Rate limited. Retrying in 60 sec")
                    time.sleep(60)

                elif status == 503:
                    print(f"503 for {category}, {year} at start={start}, attempt {attempt+1}/{MAX_RETRIES}")
                    print("Service unavailable. Retrying in 90 sec ")
                    time.sleep(90)

                else:
                    print(f"HTTP error for {category}, {year} at start={start}, attempt {attempt+1}/{MAX_RETRIES}: {e}")
                    print("Retrying in 30 sec")
                    time.sleep(30)

            except ET.ParseError:
                print(f"Parse error for {category},{year} at start={start}, attempt {attempt+1}/{MAX_RETRIES}")
                print("Retrying in 20 sec")
                time.sleep(20)

            except requests.RequestException as e:
                print(f"Request error for {category}, {year} at start={start}, attempt {attempt+1}/{MAX_RETRIES}: {e}")
                print("Retrying in 30 sec")
                time.sleep(30)

            except Exception as e:
                print(f"Error for {category}, {year} at start={start}, attempt {attempt+1}/{MAX_RETRIES}: {e}")
                print("Retrying in 20 sec.")
                time.sleep(20)

        # skip if unsucessful run
        if not success:
            print(f"Skipping batch for {category}, {year} at start={start} after {MAX_RETRIES} failed attempts.")
            start += current_batch_size
            continue

        entries = root.findall('atom:entry',ns)
        if not entries:
            break

        for entry in entries:
            title = entry.find('atom:title', ns)
            abstract = entry.find('atom:summary', ns)

            if title is not None and abstract is not None:
                records.append({
                    'title': title.text.strip().replace('\n', ' '),
                    'abstract': abstract.text.strip().replace('\n', ' ')
                })

            if len(records) >=max_results:
                break

        start += current_batch_size

    return records

In [6]:
# empty list for the dataset

records = []

In [7]:
# collection process

records = []

for category, info in taxonomy.items():
    print(f"Scraping: {category}")

    for year in years:
        print(f"  Year: {year}")

        papers = scrape_arxiv(category, year, max_results=500)

        for paper in papers:
            records.append({
                'category': category,
                'field': info['field'],
                'discipline': info['discipline'],
                'year': year,
                'title': paper['title'],
                'abstract': paper['abstract']
            })
        print(f"    {len(papers)} collected!")

print(f"\nTotal: {len(records)}")

Scraping: cs.LG
  Year: 2021
    500 collected!
  Year: 2022
    500 collected!
  Year: 2023
    500 collected!
  Year: 2024
    500 collected!
  Year: 2025
    500 collected!
Scraping: cs.CV
  Year: 2021
    500 collected!
  Year: 2022
    500 collected!
  Year: 2023
    500 collected!
  Year: 2024
    500 collected!
  Year: 2025
    500 collected!
Scraping: cs.AI
  Year: 2021
    500 collected!
  Year: 2022
    500 collected!
  Year: 2023
    500 collected!
  Year: 2024
    500 collected!
  Year: 2025
    500 collected!
Scraping: cs.CL
  Year: 2021
    500 collected!
  Year: 2022
    500 collected!
  Year: 2023
    500 collected!
  Year: 2024
    500 collected!
  Year: 2025
    500 collected!
Scraping: cs.IT
  Year: 2021
    500 collected!
  Year: 2022
    500 collected!
  Year: 2023
    500 collected!
  Year: 2024
    500 collected!
  Year: 2025
    500 collected!
Scraping: cs.RO
  Year: 2021
    500 collected!
  Year: 2022
    500 collected!
  Year: 2023
    500 collected!
  Year: 

### Cleaning

In [8]:
# convert to dataframe and examine

df = pd.DataFrame(records)
df.head()
df.shape

(79216, 6)

In [19]:
print(df.info())

# shape
print("Shape:", df.shape)

# columns
print("\nColumns:", df.columns.tolist())

# preview
print("\nHead:")
print(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 79215 entries, 0 to 79215
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   category         79215 non-null  object
 1   field            79215 non-null  object
 2   discipline       79215 non-null  object
 3   year             79215 non-null  int64 
 4   title            79215 non-null  object
 5   abstract         79215 non-null  object
 6   abstract_length  79215 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 4.8+ MB
None
Shape: (79215, 7)

Columns: ['category', 'field', 'discipline', 'year', 'title', 'abstract', 'abstract_length']

Head:
  category             field               discipline  year  \
0    cs.LG  Machine Learning  Artificial Intelligence  2021   
1    cs.LG  Machine Learning  Artificial Intelligence  2021   
2    cs.LG  Machine Learning  Artificial Intelligence  2021   
3    cs.LG  Machine Learning  Artificial Intelligence  2021   
4  

In [20]:
df = df.drop(columns=["year"])

In [21]:
print(df.shape)
print(df.columns)

(79215, 6)
Index(['category', 'field', 'discipline', 'title', 'abstract',
       'abstract_length'],
      dtype='object')


In [10]:
# check for duplicates

print(df.isnull().sum())
print(f"\nDuplicates: {df.duplicated().sum()}")

category      0
field         0
discipline    0
year          0
title         0
abstract      0
dtype: int64

Duplicates: 1


In [11]:
df = df.drop_duplicates()
print(f"Remaining records: {len(df)}")

Remaining records: 79215


In [23]:
# list unique disciplies

print(df['discipline'].unique())
print(df['field'].unique())


['Artificial Intelligence' 'Computer Imaging and Vision' 'Communication'
 'Robotics' 'Cybersecurity' 'Human-Computer Interaction'
 'Theory of Computation' 'Distributed Systems' 'Social Computing'
 'Computer Networks' 'Software Engineering' 'Information Retrieval'
 'World Wide Web' 'Multimedia' 'Database Systems' 'Computational Science'
 'Computer Programming' 'Computer Graphics' 'Computer Hardware'
 'Emerging Technologies' 'Theoretical Computer Science' 'Computer Systems']
['Machine Learning' 'Computer Vision' 'General Artificial Intelligence'
 'Natural Language Processing' 'Information Theory' 'General Robotics'
 'Cryptography' 'General Human-Computer Interaction'
 'Data Structures and Algorithms' 'Distributed Computer Systems'
 'Computers and Society' 'General Computer Networks'
 'General Software Engineering' 'General Information Retrieval'
 'Social Networks' 'Sound and Signal Processing' 'Formal Logic'
 'Neural Networks' 'Discrete Mathematics' 'Game Theory'
 'Computational Complexi

In [25]:
# clean text

df['abstract'] = df['abstract'].str.strip()
df['title'] = df['title'].str.strip()

In [15]:
# abstract lenght & analysis

df['abstract_length'] = df['abstract'].str.split().str.len()

print(df['abstract_length'].describe())

count    79215.000000
mean       174.365916
std         53.089616
min          4.000000
25%        139.000000
50%        174.000000
75%        211.000000
max        496.000000
Name: abstract_length, dtype: float64


In [26]:
df['title_length'] = df['title'].str.split().str.len()

print(df['title_length'].describe())

count    79215.000000
mean         9.666351
std          3.306094
min          1.000000
25%          7.000000
50%          9.000000
75%         12.000000
max         35.000000
Name: title_length, dtype: float64


In [27]:
# print random abstracts

print("Random abstracts: \n")
for a in df['abstract'].sample(10, random_state=42):
    print(a[:300], "\n")

Random abstracts: 

The semi-random graph process is an adaptive random graph process in which an online algorithm is initially presented an empty graph on $n$ vertices. In each round, a vertex $u$ is presented to the algorithm independently and uniformly at random. The algorithm then adaptively selects a vertex $v$, a 

Multiagent reinforcement learning algorithms have not been widely adopted in large scale environments with many agents as they often scale poorly with the number of agents. Using mean field theory to aggregate agents has been proposed as a solution to this problem. However, almost all previous metho 

Various numerical methods used for solving partial differential equations (PDE) result in tridiagonal systems. Solving tridiagonal systems on distributed-memory environments is not straightforward, and often requires significant amount of communication. In this article, we present a novel distribute 

Let $M$ be a Seifert fiber space with non-abelian fundamental group and

In [28]:
print("Random titles: \n")
for a in df['title'].sample(10, random_state=42):
    print(a[:300], "\n")

Random titles: 

Building Hamiltonian Cycles in the Semi-Random Graph Process in Less Than $2n$ Rounds 

Decentralized Mean Field Games 

A Distributed-memory Tridiagonal Solver Based on a Specialised Data Structure Optimised for CPU and GPU Architectures 

Small $\text{PSL}(2, \mathbb{F})$ representations of Seifert fiber space groups 

Twenty-Five Years of MIR Research: Achievements, Practices, Evaluations, and Future Challenges 

On Explaining Recommendations with Large Language Models: A Review 

Channel Charting-assisted Non-orthogonal Pilot Allocation for Uplink XL-MIMO Transmission 

Symbolic Specialization of Rewriting Logic Theories with Presto 

Revisiting Graph Homophily Measures 

HPRN: Holistic Prior-embedded Relation Network for Spectral Super-Resolution 



In [30]:
# remove special characters

df['title'] = df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

print("Random titles: \n")
for a in df['title'].sample(10, random_state=42):
    print(a[:300], "\n")

Random titles: 

Building Hamiltonian Cycles in the SemiRandom Graph Process in Less Than 2n Rounds 

Decentralized Mean Field Games 

A Distributedmemory Tridiagonal Solver Based on a Specialised Data Structure Optimised for CPU and GPU Architectures 

Small textPSL2 mathbbF representations of Seifert fiber space groups 

TwentyFive Years of MIR Research Achievements Practices Evaluations and Future Challenges 

On Explaining Recommendations with Large Language Models A Review 

Channel Chartingassisted Nonorthogonal Pilot Allocation for Uplink XLMIMO Transmission 

Symbolic Specialization of Rewriting Logic Theories with Presto 

Revisiting Graph Homophily Measures 

HPRN Holistic Priorembedded Relation Network for Spectral SuperResolution 



In [31]:
# remove irrelevant abstracts

noise_keywords = ['withdrawn', 'this paper has been withdrawn', 'this volume contains']
mask = df['abstract'].str.contains('|'.join(noise_keywords), case=False, na=False)
print(f"Irrelevant abstracts: {mask.sum()}")

# remove noise
df = df[~mask]

print(f"Remaining abstracts: {len(df)}")

Irrelevant abstracts: 74
Remaining abstracts: 79141


In [ ]:
# remove very short abstracts

df = df[df['abstract_length'] >= 50]

print(f"Remaining abstracts: {len(df)}")

In [33]:
# lowercase text

df['abstract'] = df['abstract'].str.lower()
df['title'] = df['title'].str.lower()

In [34]:
# check for empty strings

print((df['abstract'].str.strip() == "").sum())
print((df['title'].str.strip() == "").sum())

0
0


In [35]:
print(df['category'].value_counts())

category
cs.LG    2500
cs.CV    2500
cs.AI    2500
cs.CL    2500
cs.IT    2500
cs.RO    2500
cs.CR    2500
cs.DS    2500
cs.SD    2500
cs.NI    2500
cs.SI    2500
cs.IR    2500
cs.MM    2500
cs.DB    2500
cs.DM    2500
cs.NE    2500
cs.CG    2500
cs.AR    2500
cs.CY    2499
cs.HC    2499
cs.CE    2499
cs.MA    2499
cs.DC    2498
cs.GT    2497
cs.CC    2497
cs.SE    2494
cs.PL    2486
cs.ET    2474
cs.LO    2468
cs.FL    2301
cs.DL    2254
cs.PF    2176
Name: count, dtype: int64


In [37]:
# count per discipline
print("Discipline counts:\n")
print(df['discipline'].value_counts())

print("\n" + "-"*40 + "\n")

# count per field
print("Field counts:\n")
print(df['field'].value_counts())

Discipline counts:

discipline
Artificial Intelligence         17464
Theory of Computation            7497
Multimedia                       5000
World Wide Web                   4754
Communication                    2500
Computer Imaging and Vision      2500
Information Retrieval            2500
Cybersecurity                    2500
Robotics                         2500
Computer Networks                2500
Computer Graphics                2500
Computer Hardware                2500
Database Systems                 2500
Human-Computer Interaction       2499
Social Computing                 2499
Computational Science            2499
Distributed Systems              2498
Software Engineering             2494
Computer Programming             2486
Emerging Technologies            2474
Theoretical Computer Science     2301
Computer Systems                 2176
Name: count, dtype: int64

----------------------------------------

Field counts:

field
Machine Learning                        250

In [40]:
# save
import os
os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)
# save dataset
df.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/arxiv_abstracts_and_titles.csv",
    index=False
)
print("Saved")

Saved
